# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import itertools
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_scaled = pd.read_csv('../data/dayofweek.csv')
df['dayofweek'] = df_scaled['dayofweek']

df.head()

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4


In [3]:
y = df.dayofweek
X = df.drop('dayofweek', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [17]:
clf = GridSearchCV(SVC(random_state=21, probability=True), {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None]
})

In [18]:
clf.fit(X_train, y_train)

GridSearchCV(estimator=SVC(probability=True, random_state=21),
             param_grid={'C': [0.01, 0.1, 1, 1.5, 5, 10],
                         'class_weight': ['balanced', None],
                         'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf', 'sigmoid']})

In [25]:
svm_cv_df = pd.DataFrame(clf.cv_results_)

In [26]:
svm_cv_df[['param_C', 'param_class_weight', 'param_gamma', 'param_kernel', 'mean_test_score', 'rank_test_score']].sort_values('rank_test_score')

,param_C,param_class_weight,param_gamma,param_kernel,mean_test_score,rank_test_score
70,10,None,auto,rbf,0.875368,1
64,10,balanced,auto,rbf,0.866457,2
58,5,None,auto,rbf,0.802677,3
52,5,balanced,auto,rbf,0.796737,4
60,10,balanced,scale,linear,0.740300,5
...,...,...,...,...,...,...
38,1.5,balanced,scale,sigmoid,0.142476,68
17,0.1,balanced,auto,sigmoid,0.120983,69
65,10,balanced,auto,sigmoid,0.104654,70
41,1.5,balanced,auto,sigmoid,0.100204,71


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [4]:
dt_clf = GridSearchCV(DecisionTreeClassifier(random_state=21), {
    'max_depth': np.arange(1, 50),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini'] 
})

In [5]:
dt_clf.fit(X_train, y_train)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=21),
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49])})

In [6]:
dt_cv_df = pd.DataFrame(dt_clf.cv_results_)

dt_cv_df[['param_max_depth', 'param_class_weight', 'param_criterion', 'mean_test_score', 'rank_test_score']].sort_values('rank_test_score')

,param_max_depth,param_class_weight,param_criterion,mean_test_score,rank_test_score
69,21,balanced,gini,0.873865,1
73,25,balanced,gini,0.873854,2
70,22,balanced,gini,0.872378,3
97,49,balanced,gini,0.872372,4
71,23,balanced,gini,0.872372,4
...,...,...,...,...,...
51,3,balanced,gini,0.373906,192
147,1,None,gini,0.355330,193
98,1,None,entropy,0.355330,193
49,1,balanced,gini,0.286358,195


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [7]:
rf_clf = GridSearchCV(RandomForestClassifier(random_state=21), {
    'n_estimators': [5, 10, 50, 100],
    'max_depth': np.arange(1, 50),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini'] 
}, cv=5)

In [8]:
rf_clf.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=21),
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]),
                         'n_estimators': [5, 10, 50, 100]})

In [9]:
rf_cv_df = pd.DataFrame(rf_clf.cv_results_)

rf_scores = rf_cv_df[['param_n_estimators', 'param_max_depth', 'param_class_weight', 'param_criterion', 'mean_test_score', 'rank_test_score']].sort_values('rank_test_score')

rf_scores

,param_n_estimators,param_max_depth,param_class_weight,param_criterion,mean_test_score,rank_test_score
95,100,24,balanced,entropy,0.904293,1
115,100,29,balanced,entropy,0.904290,2
698,50,28,None,gini,0.904290,2
314,50,30,balanced,gini,0.903549,4
711,100,31,None,gini,0.903547,5
...,...,...,...,...,...,...
392,5,1,None,entropy,0.353832,780
4,5,2,balanced,entropy,0.353110,781
200,5,2,balanced,gini,0.346419,782
196,5,1,balanced,gini,0.283390,783


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [4]:
grid = {
    'n_estimators': [5, 10, 50, 100],
    'max_depth': np.arange(1, 50),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini'] 
}
keys = grid.keys()
values = grid.values()

results = []

combinations = list(itertools.product(*values))

for combination in tqdm(combinations):
    params = dict(zip(keys, combination))
    model = RandomForestClassifier(random_state=21, n_jobs=-1, **params)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)

    results.append({
        **params,
        'mean_accuracy': scores.mean(),
        'std_accuracy': scores.std()
    })

  0%|          | 0/784 [00:00<?, ?it/s]

In [5]:
result_df = pd.DataFrame(results)

result_df.sort_values('mean_accuracy', ascending=False)

,n_estimators,max_depth,class_weight,criterion,mean_accuracy,std_accuracy
680,100,24,balanced,entropy,0.904293,0.012361
503,50,28,None,gini,0.904290,0.010961
700,100,29,balanced,entropy,0.904290,0.012156
509,50,30,balanced,gini,0.903549,0.012056
711,100,31,None,gini,0.903547,0.014380
...,...,...,...,...,...,...
2,5,1,None,entropy,0.353832,0.016467
4,5,2,balanced,entropy,0.353110,0.021165
5,5,2,balanced,gini,0.346419,0.029749
1,5,1,balanced,gini,0.283390,0.011062


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [6]:
model = RandomForestClassifier(random_state=21, n_estimators=100, max_depth=24, class_weight='balanced', criterion='entropy')

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=24, random_state=21)

In [7]:
model.score(X_test, y_test)

0.9260355029585798